# BlackjackVAI — Entrenamiento v6

**Modelo base:** `yolov8m-seg.pt` (pesos COCO — sin conocimiento previo de cartas)  
**Dataset:** BlackjackVAI V4 · 80/20 train/val  
**GPU:** Local (RTX 4060 / cualquier CUDA)  
**Tracking:** MLflow  

### Diferencias respecto a v5
| Aspecto | v5 | v6 |
|---------|----|----|  
| Modelo base | best.pt v4 | **yolov8m-seg.pt (COCO)** |
| Objetivo | Fine-tuning | **Entrenamiento limpio desde COCO** |
| lr0 | 0.0005 | **0.01** (estándar YOLO) |
| Epochs | 80 | **150** (más tiempo para aprender cartas) |
| Aug geométrico | v4 params | **v4 params (idéntico)** |
| Aug calidad | v5 albumentations | **v5 albumentations (idéntico)** |
| Dataset | V4 × 3 imágenes | **V4 × 3 imágenes (idéntico)** |

### Por qué entrenar desde COCO
El modelo COCO ya conoce formas, bordes y texturas generales.  
Al no partir de pesos de cartas previos, el v6 aprende los patrones del dataset V4  
sin ningún sesgo heredado de v3/v4/v5 — útil para comparar de forma justa.

## 0. Configuración

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# MLflow
MLFLOW_EXPERIMENT = "blackjackvai-v6"
MLFLOW_DB = "sqlite:///" + os.path.abspath("mlflow.db")
os.environ["MLFLOW_TRACKING_URI"]    = MLFLOW_DB
os.environ["MLFLOW_EXPERIMENT_NAME"] = MLFLOW_EXPERIMENT

# Roboflow — mismo dataset V4 que v4/v5
ROBOFLOW_API_KEY   = os.environ["ROBOFLOW_API_KEY"]
ROBOFLOW_WORKSPACE = "javiers-workspace-q8mnr"
ROBOFLOW_PROJECT   = "blackjackvai"
ROBOFLOW_VERSION   = 4

from pathlib import Path
BASE_DIR    = Path(".").resolve()
DATASET_DIR = BASE_DIR / f"BlackjackVAI-{ROBOFLOW_VERSION}"
WORKDIR     = BASE_DIR / "training_runs"
MODELS_DIR  = BASE_DIR / "models"

RUN_NAME    = "yolo8m_seg_v6"
MODEL_BASE  = "yolov8m-seg.pt"   # ← pesos COCO, sin conocimiento previo de cartas
EPOCHS      = 150
IMGSZ       = 640
BATCH       = 8
PATIENCE    = 30
SEED        = 42
SAVE_PERIOD = 10
RESUME      = True               # True: reanuda si existe last.pt

DATA_YAML = DATASET_DIR / "data.yaml"
RUN_DIR   = WORKDIR / "runs" / RUN_NAME
BEST_PT   = RUN_DIR / "weights" / "best.pt"
LAST_PT   = RUN_DIR / "weights" / "last.pt"

WORKDIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Dataset dir         : {DATASET_DIR}")
print(f"Modelo base         : {MODEL_BASE}  ← COCO pretrained, sin cartas previas")
print(f"Run dir             : {RUN_DIR}")
print(f"MLflow DB           : {MLFLOW_DB}")
print(f"MLflow experiment   : {MLFLOW_EXPERIMENT}")
print(f"Resume              : {RESUME}")

## 1. Dependencias y GPU

In [ ]:
import torch

DEVICE = 0 if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    vram_gb = gpu.total_memory / 1e9
    print(f"GPU : {gpu.name}  |  VRAM: {vram_gb:.1f} GB")
    if vram_gb < 6:
        print("AVISO: menos de 6 GB VRAM — considera BATCH=4")
else:
    print("AVISO: sin GPU — el entrenamiento será muy lento")

import ultralytics, mlflow
print(f"Ultralytics {ultralytics.__version__} | MLflow {mlflow.__version__} | PyTorch {torch.__version__}")

try:
    import albumentations as A
    print(f"Albumentations {A.__version__} ✓")
except ImportError:
    print("AVISO: albumentations no instalado — ejecuta: pip install albumentations")

## 2. Dataset V4 desde Roboflow

In [ ]:
if DATA_YAML.exists():
    print(f"Dataset ya existe en {DATASET_DIR} — saltando descarga.")
else:
    print("Descargando dataset V4 desde Roboflow...")
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
    dataset = project.version(ROBOFLOW_VERSION).download("yolov8", location=str(DATASET_DIR))
    print(f"Dataset descargado en: {dataset.location}")

## 3. Split 80/20 — fusionar test en train

In [ ]:
import shutil

test_img_dir  = DATASET_DIR / "test"  / "images"
test_lbl_dir  = DATASET_DIR / "test"  / "labels"
train_img_dir = DATASET_DIR / "train" / "images"
train_lbl_dir = DATASET_DIR / "train" / "labels"

merged = 0
if test_img_dir.exists():
    for img in test_img_dir.glob("*.*"):
        dest = train_img_dir / img.name
        if not dest.exists():
            shutil.copy2(img, dest)
            merged += 1
    for lbl in test_lbl_dir.glob("*.txt"):
        dest = train_lbl_dir / lbl.name
        if not dest.exists():
            shutil.copy2(lbl, dest)
    print(f"Fusionadas {merged} imágenes de test → train")
else:
    print("Carpeta test/ no encontrada — split ya aplicado")

for split in ["train", "val", "test"]:
    d = DATASET_DIR / split / "images"
    if d.exists():
        print(f"  {split}: {len(list(d.glob('*.*')))} imágenes")

## 4. Preparar data.yaml

In [ ]:
import yaml

with open(DATA_YAML) as f:
    data = yaml.safe_load(f)

data["path"] = str(DATASET_DIR.resolve())

for key in list(data.keys()):
    if key == "valid":
        val = data.pop(key)
        data["val"] = str(val).replace("valid/", "val/").replace("/valid/", "/val/")

valid_dir = DATASET_DIR / "valid"
val_dir   = DATASET_DIR / "val"
if valid_dir.exists() and not val_dir.exists():
    shutil.copytree(valid_dir, val_dir)
    print("Carpeta 'valid' copiada a 'val'")

data.pop("test", None)

with open(DATA_YAML, "w") as f:
    yaml.safe_dump(data, f, sort_keys=False, allow_unicode=True)

names = data["names"] if isinstance(data["names"], list) else list(data["names"].values())
print(f"Clases ({data['nc']}): {names[:8]} ...")
print(f"Train : {data.get('train')}")
print(f"Val   : {data.get('val')}")

for cache in DATASET_DIR.rglob("*.cache"):
    cache.unlink()
    print(f"Cache eliminada: {cache}")

## 5. Limpiar etiquetas mixtas detect/segment

In [ ]:
def fix_labels(label_dir: Path) -> int:
    n_fixed = 0
    for lbl in label_dir.glob("*.txt"):
        lines = lbl.read_text(encoding="utf-8").splitlines()
        seg_lines = [
            ln for ln in lines
            if ln.strip() and
               len(ln.strip().split()) > 5 and
               (len(ln.strip().split()) - 1) % 2 == 0
        ]
        if len(seg_lines) != len([l for l in lines if l.strip()]):
            lbl.write_text("\n".join(seg_lines), encoding="utf-8")
            n_fixed += 1
    return n_fixed

for split in ["train", "val"]:
    lbl_dir = DATASET_DIR / split / "labels"
    if lbl_dir.exists():
        n = fix_labels(lbl_dir)
        print(f"{split}: {n} ficheros corregidos")
print("Labels OK.")

## 6. Augmentation de calidad (v5) — blur, JPEG, ruido, baja luz

Genera **2 copias degradadas** por imagen original → **3× más datos** de entrenamiento.  
Las etiquetas se copian tal cual (las degradaciones no desplazan las cartas).  
Si ya se generaron en v5 los reutiliza directamente.

In [ ]:
import cv2
import numpy as np
import albumentations as A
import random

random.seed(SEED)
np.random.seed(SEED)

pipeline_blur = A.Compose([
    A.OneOf([
        A.GaussianBlur(blur_limit=(3, 9), p=1.0),
        A.MotionBlur(blur_limit=(5, 15), p=1.0),
        A.MedianBlur(blur_limit=7, p=1.0),
    ], p=1.0),
    A.GaussNoise(var_limit=(10, 60), p=0.5),
])

pipeline_quality = A.Compose([
    A.ImageCompression(quality_lower=20, quality_upper=55, p=1.0),
    A.OneOf([
        A.RandomBrightnessContrast(brightness_limit=(-0.4, -0.1),
                                   contrast_limit=(-0.3, 0.1), p=1.0),
        A.RandomGamma(gamma_limit=(40, 80), p=1.0),
    ], p=0.8),
    A.GaussNoise(var_limit=(20, 80), p=0.6),
])

PIPELINES = [
    ("_aug_blur",    pipeline_blur),
    ("_aug_quality", pipeline_quality),
]

img_dir = DATASET_DIR / "train" / "images"
lbl_dir = DATASET_DIR / "train" / "labels"

orig_imgs = [p for p in img_dir.glob("*.*")
             if not any(tag in p.stem for tag in ["_aug_blur", "_aug_quality"])]

generated = 0
skipped   = 0

for img_path in orig_imgs:
    img = cv2.imread(str(img_path))
    if img is None:
        continue
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    lbl_src = lbl_dir / f"{img_path.stem}.txt"

    for suffix, pipeline in PIPELINES:
        out_img = img_dir / f"{img_path.stem}{suffix}{img_path.suffix}"
        out_lbl = lbl_dir / f"{img_path.stem}{suffix}.txt"

        if out_img.exists():
            skipped += 1
            continue

        aug = pipeline(image=img_rgb)["image"]
        cv2.imwrite(str(out_img), cv2.cvtColor(aug, cv2.COLOR_RGB2BGR),
                    [cv2.IMWRITE_JPEG_QUALITY, 92])
        if lbl_src.exists():
            shutil.copy2(lbl_src, out_lbl)
        generated += 1

total = len(list(img_dir.glob("*.*")))
print(f"Generadas  : {generated} imágenes nuevas")
print(f"Omitidas   : {skipped} (ya existían)")
print(f"Total train: {total} imágenes  ({len(orig_imgs)} orig + {total - len(orig_imgs)} aug)")

## 7. Visualización — muestras del dataset

In [ ]:
import matplotlib.pyplot as plt
import random

random.seed(SEED)
CMAP = plt.cm.get_cmap("tab20", len(names))

def bgr_color(cls_id):
    r, g, b, _ = CMAP(cls_id % 20)
    return (int(b*255), int(g*255), int(r*255))

def draw_sample(img_path: Path, lbl_path: Path):
    img = cv2.imread(str(img_path))
    if img is None: return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    if not lbl_path.exists(): return img
    for ln in lbl_path.read_text(encoding="utf-8").splitlines():
        toks = ln.strip().split()
        if not toks or len(toks) < 7: continue
        cls_id = int(float(toks[0]))
        coords = np.array(list(map(float, toks[1:])), dtype=np.float32).reshape(-1, 2)
        coords[:, 0] *= w; coords[:, 1] *= h
        pts = coords.astype(np.int32)
        color = bgr_color(cls_id)
        if len(pts) >= 3:
            ov = img.copy()
            cv2.fillPoly(ov, [pts], color)
            img = cv2.addWeighted(ov, 0.25, img, 0.75, 0)
            cv2.polylines(img, [pts], True, color, 2)
        x0, y0 = pts.min(axis=0)
        label = names[cls_id] if cls_id < len(names) else f"cls_{cls_id}"
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        y0c = max(y0, th + 4)
        cv2.rectangle(img, (x0, y0c-th-4), (x0+tw+2, y0c+2), color, -1)
        cv2.putText(img, label, (x0+1, y0c-1), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 1)
    return img

train_imgs = sorted((DATASET_DIR / "train" / "images").glob("*.*"))
# Mostrar solo originales (sin aug)
orig_only = [p for p in train_imgs if not any(t in p.stem for t in ["_aug_blur", "_aug_quality"])]
samples   = random.sample(orig_only, min(9, len(orig_only)))

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
for ax, img_path in zip(axes.flatten(), samples):
    lbl_path = img_path.parent.parent / "labels" / f"{img_path.stem}.txt"
    ann = draw_sample(img_path, lbl_path)
    if ann is not None:
        ax.imshow(ann)
        ax.set_title(img_path.name, fontsize=8)
    ax.axis("off")
plt.suptitle("Dataset V4 — muestras originales con anotaciones", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 8. Configurar MLflow

In [ ]:
from ultralytics import settings as yolo_settings

yolo_settings.update({"mlflow": True})

print(f"YOLO mlflow activo  : {yolo_settings.get('mlflow')}")
print(f"MLflow URI          : {os.environ['MLFLOW_TRACKING_URI']}")
print(f"MLflow experiment   : {os.environ['MLFLOW_EXPERIMENT_NAME']}")
print("Todo listo — ejecuta la celda de entrenamiento.")

## 9. Entrenamiento desde yolov8m-seg.pt (COCO)

### Augmentation combinado v4 + v5

| Parámetro | Valor | Razón |
|-----------|-------|-------|
| `scale` | 0.85 | Cartas desde 15% hasta 185% — distancia variable |
| `mosaic` | 1.0 | 4 imágenes combinadas → más variedad de escenas |
| `copy_paste` | 0.60 | Pega cartas sobre otros fondos |
| `erasing` | 0.50 | Oclusión agresiva (manos, fichas) |
| `perspective` | 0.001 | Ángulo de la webcam respecto a la mesa |
| `shear` | 8.0 | Perspectiva lateral agresiva |
| `degrees` | 20.0 | Rotación — cartas no siempre alineadas |
| `hsv_s` | 0.90 | Saturación — distintas iluminaciones |
| `hsv_v` | 0.60 | Brillo — sombras y sobreexposición |
| `mixup` | 0.25 | Mezcla entre imágenes |
| `close_mosaic` | 20 | Epochs con mosaic antes de desactivarlo |

### lr0 más alto que v5
Al partir de COCO (no de cartas previas), se usa `lr0=0.01` (default YOLO)  
para permitir que el modelo aprenda las características de las cartas desde cero.

In [ ]:
from ultralytics import YOLO

resuming = RESUME and LAST_PT.exists()

if resuming:
    print(f"Reanudando desde: {LAST_PT}")
    model = YOLO(str(LAST_PT))
else:
    print(f"Entrenando desde base COCO: {MODEL_BASE}")
    model = YOLO(MODEL_BASE)

train_kwargs = dict(
    data         = str(DATA_YAML),
    task         = "segment",
    epochs       = EPOCHS,
    imgsz        = IMGSZ,
    batch        = BATCH,
    device       = DEVICE,
    workers      = 4,
    seed         = SEED,
    pretrained   = True,
    amp          = True,
    cache        = False,
    cos_lr       = True,
    # lr0 no especificado → YOLO lo calcula automáticamente según batch/optimizador
    lrf          = 0.01,
    warmup_epochs= 5,           # más warmup al empezar desde COCO
    patience     = PATIENCE,
    save_period  = SAVE_PERIOD,
    plots        = True,
    save         = True,
    project      = str(WORKDIR / "runs"),
    name         = RUN_NAME,
    exist_ok     = True,
    rect         = False,       # letterbox — sin distorsión de cartas
    # ── Augmentation geométrico (v4) ────────────────────────────────
    hsv_h        = 0.02,
    hsv_s        = 0.90,
    hsv_v        = 0.60,
    degrees      = 20.0,
    translate    = 0.20,
    scale        = 0.85,
    shear        = 8.0,
    perspective  = 0.001,
    flipud       = 0.25,
    fliplr       = 0.50,
    mosaic       = 1.0,
    mixup        = 0.25,
    copy_paste   = 0.60,
    erasing      = 0.50,
    close_mosaic = 20,
    overlap_mask = True,
    mask_ratio   = 4,
)

if resuming:
    results = model.train(resume=True)
else:
    results = model.train(**train_kwargs)

RUN_DIR_FINAL = Path(model.trainer.save_dir)
BEST_PT_FINAL = RUN_DIR_FINAL / "weights" / "best.pt"
LAST_PT_FINAL = RUN_DIR_FINAL / "weights" / "last.pt"

print(f"\nEntrenamiento completado.")
print(f"best.pt → {BEST_PT_FINAL}")
print(f"last.pt → {LAST_PT_FINAL}")

## 10. Curvas de entrenamiento

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

results_csv = RUN_DIR_FINAL / "results.csv"
if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()

    fig, axes = plt.subplots(1, 3, figsize=(18, 4))

    for col, lbl in [("train/box_loss", "train"), ("val/box_loss", "val")]:
        if col in df.columns: axes[0].plot(df["epoch"], df[col], label=lbl)
    axes[0].set_title("Box Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()

    for col, lbl in [("metrics/mAP50(B)", "mAP50"), ("metrics/mAP50-95(B)", "mAP50-95")]:
        if col in df.columns: axes[1].plot(df["epoch"], df[col], label=lbl)
    axes[1].set_title("mAP — Bounding Box"); axes[1].set_xlabel("Epoch"); axes[1].legend()

    for col, lbl in [("metrics/mAP50(M)", "mAP50"), ("metrics/mAP50-95(M)", "mAP50-95")]:
        if col in df.columns: axes[2].plot(df["epoch"], df[col], label=lbl)
    axes[2].set_title("mAP — Máscara"); axes[2].set_xlabel("Epoch"); axes[2].legend()

    plt.suptitle("Curvas de entrenamiento — BlackjackVAI V6", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print("results.csv no encontrado — ejecuta la celda de entrenamiento primero.")

## 11. Evaluación sobre validación + log MLflow

In [ ]:
import mlflow
import re
from ultralytics import YOLO
from datetime import datetime

def sanitize(key):
    return re.sub(r'[^a-zA-Z0-9_\-\. :/]', '_', key)

model = YOLO(str(BEST_PT_FINAL))

m = model.val(
    data=str(DATA_YAML),
    split="val",
    imgsz=IMGSZ,
    conf=0.5,
    iou=0.45,
    plots=True,
    save_json=True,
)

print(f"\nmAP50      (box) : {m.box.map50:.4f}")
print(f"mAP50-95   (box) : {m.box.map:.4f}")
print(f"mAP50      (mask): {m.seg.map50:.4f}")
print(f"mAP50-95   (mask): {m.seg.map:.4f}")

mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
mlflow.set_experiment(MLFLOW_EXPERIMENT)

with mlflow.start_run(run_name=f"val_eval_{datetime.now():%Y%m%d_%H%M}"):
    mlflow.log_metrics({
        "val/mAP50_box":     m.box.map50,
        "val/mAP50-95_box":  m.box.map,
        "val/mAP50_mask":    m.seg.map50,
        "val/mAP50-95_mask": m.seg.map,
        "val/precision":     m.box.mp,
        "val/recall":        m.box.mr,
    })

    for k, v in m.results_dict.items():
        try:
            mlflow.log_metric(f"val_{sanitize(k)}", float(v))
        except (TypeError, ValueError):
            pass

    mlflow.log_params({
        "base_model":      "yolov8m-seg.pt (COCO)",
        "lr0":             "auto",
        "scale":           0.85,
        "copy_paste":      0.60,
        "erasing":         0.50,
        "perspective":     0.001,
        "shear":           8.0,
        "mosaic":          1.0,
        "mixup":           0.25,
        "quality_aug":     "blur+noise+jpeg+lowlight",
        "train_images":    "x3 (orig+blur+quality)",
        "imgsz_train":     IMGSZ,
        "imgsz_infer":     1280,
        "split":           "80/20 (sin test)",
        "dataset_ver":     4,
        "epochs":          EPOCHS,
        "warmup_epochs":   5,
    })

    mlflow.log_artifact(str(BEST_PT_FINAL), artifact_path="model")
    mlflow.log_artifacts(str(RUN_DIR_FINAL), artifact_path="train_artifacts")

    mlflow.set_tags({
        "stage":        "val_evaluation",
        "dataset":      "BlackjackVAI-V4-x3aug",
        "classes":      54,
        "backbone":     "yolov8m-seg",
        "base_weights": "COCO",
        "aug_strategy": "distance+quality_degradation",
    })

    print("\nLoggeado en MLflow.")

## 12. Test de robustez — predicciones bajo degradación

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import albumentations as A
import random
from ultralytics import YOLO

random.seed(SEED)
rob_model = YOLO(str(BEST_PT_FINAL))

degradations = [
    ("Original",     lambda img: img),
    ("Blur fuerte",  lambda img: A.GaussianBlur(blur_limit=(7, 11), p=1)(image=img)["image"]),
    ("Motion blur",  lambda img: A.MotionBlur(blur_limit=13, p=1)(image=img)["image"]),
    ("JPEG q=15",    lambda img: A.ImageCompression(quality_lower=10, quality_upper=20, p=1)(image=img)["image"]),
    ("Baja luz",     lambda img: A.RandomBrightnessContrast(brightness_limit=(-0.6, -0.4), contrast_limit=(-0.4, -0.2), p=1)(image=img)["image"]),
    ("Ruido+blur",   lambda img: A.Compose([A.GaussNoise(var_limit=(50, 120), p=1), A.GaussianBlur(blur_limit=(3, 7), p=1)])(image=img)["image"]),
]

val_imgs = [p for p in (DATASET_DIR / "val" / "images").glob("*.*")]
samples  = random.sample(val_imgs, min(3, len(val_imgs)))

fig, axes = plt.subplots(len(samples), len(degradations),
                         figsize=(len(degradations) * 3, len(samples) * 3))

for row, img_path in enumerate(samples):
    img_bgr = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    for col, (title, fn) in enumerate(degradations):
        deg_rgb = fn(img_rgb.copy())
        deg_bgr = cv2.cvtColor(deg_rgb, cv2.COLOR_RGB2BGR)
        res  = rob_model.predict(source=deg_bgr, imgsz=1280, conf=0.35, verbose=False)[0]
        ann  = cv2.cvtColor(res.plot(line_width=1), cv2.COLOR_BGR2RGB)
        dets = len(res.boxes) if res.boxes is not None else 0

        ax = axes[row, col]
        ax.imshow(ann)
        ax.set_title(f"{title}\n{dets} det.",
                     fontsize=8,
                     color="green" if dets > 0 else "red",
                     fontweight="bold" if row == 0 else "normal")
        ax.axis("off")

plt.suptitle("Test de robustez — detección bajo degradación (val set)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

## 13. Comparativa v5 vs v6 en MLflow

Este bloque registra ambos modelos en el mismo experimento de comparación.

In [ ]:
import mlflow
from ultralytics import YOLO

COMPARE_EXPERIMENT = "blackjackvai-comparison"
mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
mlflow.set_experiment(COMPARE_EXPERIMENT)

models_to_compare = {
    "v5_finetuned": MODELS_DIR / "best.pt",    # best.pt actual antes de v6
    "v6_from_coco": BEST_PT_FINAL,
}

for model_name, model_path in models_to_compare.items():
    if not model_path.exists():
        print(f"{model_name}: {model_path} no encontrado — saltando")
        continue

    m_model = YOLO(str(model_path))
    m_val = m_model.val(
        data=str(DATA_YAML),
        split="val",
        imgsz=IMGSZ,
        conf=0.5,
        iou=0.45,
        verbose=False,
    )

    with mlflow.start_run(run_name=model_name):
        mlflow.log_metrics({
            "mAP50_box":    m_val.box.map50,
            "mAP50-95_box": m_val.box.map,
            "mAP50_mask":   m_val.seg.map50,
            "mAP50-95_mask":m_val.seg.map,
            "precision":    m_val.box.mp,
            "recall":       m_val.box.mr,
        })
        mlflow.set_tag("model_version", model_name)
        print(f"{model_name}: mAP50(box)={m_val.box.map50:.4f}  mAP50(mask)={m_val.seg.map50:.4f}")

print(f"\nComparativa disponible en MLflow — experimento: {COMPARE_EXPERIMENT}")

## 14. Exportar modelo final a `models/`

In [ ]:
import shutil
from datetime import datetime

ts   = datetime.now().strftime("%Y%m%d_%H%M")
dest = MODELS_DIR / f"best_v6_{ts}.pt"
shutil.copy2(str(BEST_PT_FINAL), str(dest))

link = MODELS_DIR / "best.pt"
if link.exists() or link.is_symlink():
    link.unlink()
link.symlink_to(dest.name)

print(f"Modelo guardado: {dest}")
print(f"Enlace activo  : {link} → {dest.name}")
print("\nContenido de models/:")
for f in sorted(MODELS_DIR.iterdir()):
    size = f.stat().st_size / 1e6 if f.is_file() else 0
    print(f"  {f.name}  ({size:.1f} MB)")